# W12D5 ? AI Data Analyst Platform Capstone

This notebook documents the full Indian Business EDA and validates the smaller sample used to demonstrate the deployed CIA AI Data Analyst API.

**Important:** Full-dataset analysis and 5,000-row API demonstration results are separate. Recorded Amount is not net realized revenue.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data/w8d3/indian_business_sales.csv').exists():
    ROOT = ROOT.parents[1]

SOURCE = ROOT / 'data/w8d3/indian_business_sales.csv'
SAMPLE = ROOT / 'reports/w12d5/indian_business_demo_sample.csv'
print('Project root:', ROOT)
print('Full dataset exists:', SOURCE.exists())
print('Demo sample exists:', SAMPLE.exists())

## 1. Full dataset ? EDA validation

In [ ]:
df = pd.read_csv(SOURCE, low_memory=False)
print('Shape:', df.shape)
print('Date range:', df['Date'].min(), 'to', df['Date'].max())
print('Duplicate rows:', df.duplicated().sum())
print('Missing Amount:', df['Amount'].isna().sum())
print('Valid Amount observations:', df['Amount'].count())
display(df['Amount'].describe())

In [ ]:
amount = pd.to_numeric(df['Amount'], errors='coerce').dropna()
q1, q3 = amount.quantile([0.25, 0.75])
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr
print('Mean:', round(amount.mean(), 2))
print('Median:', round(amount.median(), 2))
print('Skewness:', round(amount.skew(), 3))
print('IQR upper bound:', round(upper_bound, 2))
print('Potential upper outliers:', int((amount > upper_bound).sum()))

print('\nRecorded Amount by category:')
display(df.assign(Amount=pd.to_numeric(df['Amount'], errors='coerce'))
        .groupby('Category')['Amount'].sum()
        .sort_values(ascending=False))

## 2. Deployed API demonstration sample

The original CSV exceeds the API's 10 MB upload limit. A reproducible 5,000-row sample with eight relevant columns was used for the live demonstration. Missing Amount values are retained.

In [ ]:
sample = pd.read_csv(SAMPLE)
print('Sample shape:', sample.shape)
print('Columns:', sample.columns.tolist())
print('Missing Amount:', sample['Amount'].isna().sum())
print('Duplicate rows:', sample.duplicated().sum())
print('Sample file size (bytes):', SAMPLE.stat().st_size)
assert len(sample) == 5000
assert sample.shape[1] == 8
assert SAMPLE.stat().st_size < 10 * 1024 * 1024
print('Sample validation: PASSED')

## 3. Production API demo evidence

Deployment: https://cynaris-cia-data-analyst.onrender.com

- CSV upload: HTTP 200; 5,000-row sample accepted.
- AI category query: HTTP 200; returned eight category totals and a chart. The response did not verify category averages.
- Executive summary: HTTP 200; five insights and recommendations returned.
- PowerPoint export: HTTP 200; `CIA_Executive_Summary.pptx` downloaded and opened successfully.

The API executive summary identified 271 missing Amount values and 82 duplicate rows in the sample. Duplicate rows require business review; they were not automatically removed.

## 4. Interpretation and limitations

- Full-dataset EDA findings refer to 128,975 original records; API results refer only to the 5,000-row sample.
- Missing Amount values are excluded from Amount-specific calculations; no imputation is claimed.
- Duplicate detection does not establish that records are erroneous.
- Category totals include all order statuses and must not be described as net realized revenue.
- API sessions are held in memory and may be lost when the service restarts.